<a href="https://colab.research.google.com/github/holgerlindholm/DTU12821_Site-investigations_F2026/blob/main/02_Opgaver_i_transformation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import os
import pyproj

!pip install -q pyproj
!wget -q https://raw.githubusercontent.com/holgerlindholm/DTU12821_Site-investigations_F2026/main/dk_sdfe_dnn.tif
!wget -q https://raw.githubusercontent.com/holgerlindholm/DTU12821_Site-investigations_F2026/main/dk_sdfe_gvr2016.tif

pyproj_dir = pyproj.datadir.get_data_dir()
os.replace("dk_sdfe_dnn.tif", f"{pyproj_dir}/dk_sdfe_dvr90.tif")
os.replace("dk_sdfe_gvr2016.tif", f"{pyproj_dir}/dk_sdfe_gvr2016.tif")

print(pyproj.datadir.get_data_dir())

print("✅ Workspace setup complete! You are ready to start.")

/usr/local/lib/python3.12/dist-packages/pyproj/proj_dir/share/proj
✅ Workspace setup complete! You are ready to start.


# Transformation af koordinater
Først lidt relevant information i kan få brug for. Læs eksemplerne og giv jer i kast med første opgave.

ellipsoide : GRS80

grønlandsgrid : gvr2016

danmarksgrid : dvr90

projektioner :

            cart : kartesisk
            
            utm : universal transverse mercator
            
vgridshift : vertical forskydning

---------------------------------------------------------------------------------

Eksempel 1: Data i geodætiske koordinater, men vi vil gerne have dem i kartesiske.

pipeline = "+ellps=GRS80 +proj=pipeline +step +proj=cart"

Vi bruger ellipsoide GRS80. Alt efter "+step" angiver en ny koordinat operation. I ovenstående tilfælde laver vi +proj=cart for at gå TIL kartesiske koordinater

------------------------------------------------------------------------------------------------------

Eksempel 2: Data i kartesiske koordinater, men vi vil gerne have dem i geodætiske.

pipeline = "+ellps=GRS80 +proj=pipeline +step +proj=cart +inv"

Vi bruger ellipsoide GRS80. Alt efter "+step" angiver en ny koordinat operation. I ovenstående tilfælde laver vi +proj=cart for at vælge kartesiske koordinater, og +inv for at sige, at vi går FRA kartesiske koordinater

-------------------------------------------------------------------------------------------------------
Eksempel 3: Data i UTM zone 32, vi vil gerne have dem i geodætiske koordinater

pipeline = "+ellps=GRS80 +proj=pipeline +step +proj=utm +zone=32 +inv"

Vi bruger ellipsoide GRS80. Vi kan se fra +inv operationen at vi går FRA utm zone 32 tilbage til geodætiske koordinater.

---------------------------------------------------------------------------------------------------------
Eksempel 4: Vi har koordinaterne i geodætiske, men vi vil gerne have dem med middelhavniveau som højdereference i UTM zone 32 (GVR2016)

pipeline = "+ellps=GRS80 +proj=pipeline +step +proj=vgridshift +grids=df_sdfe_gvr2016.tif +step +proj=utm +zone=32"

Vi bruger ellipsoide GRS80, og laver et vertikalt grid skifte til middelhavsniveau som højdereference med GVR2016 grid, og derefter transformerer over til UTM zone 32.

----------------------------------------------------------------------------------------------------------

### OPGAVE 1 - Vi skal til kaffemik på kaffeklubben
Vi skal have transformeret koordinaterne for fikspunkt på Kaffeklubben Ø.
Fra [Valdemar](https://valdemar.dataforsyningen.dk) har vi fundet koordinaterne i GR96, UTM zone 24N, GVR2016

| Easting    | Northing    | Ellipsoidehøjde |
| ---        | ---         | ---             |
| 604124.768 | 9299032.299 | 27.835          |

Brug *Transformer* funktionen til at lave transformationere

In [11]:
from pyproj import Transformer
kaffeklubben = [604124.768, 9299032.299, 27.835]

### 1.a Transformer Kaffeklubben fra UTM zone 24N (GRS 1980) til geodætiske koordinater stadig med ellipsiodehøjde (GRS 1980)

In [12]:
# udfyld nedenstående ukendte for at lave transformationen
ellipsoide = 'GRS80' # OBS: skal være skrevet med stort!
fra_koordinat_system = 'utm' # OBS: skal være skrevet med småt!
zone = '24'

"""
UDFYLD IKKE NEDENSTÅENDE
"""

import pyproj
print(pyproj.proj_version_str)
print(pyproj.datadir.get_data_dir())

# input til proj for at lave transformationen
pipeline = f"+ellps={ellipsoide} +proj=pipeline +step +proj={fra_koordinat_system} +zone={zone} +inv"
transform_object = Transformer.from_pipeline(pipeline)
kaffeklubben_geod_GRS = transform_object.transform(*kaffeklubben)

# print for at se resultaterne
print("Nye koordinater:")
print(f"Længdegrad (longitude):  {kaffeklubben_geod_GRS[0]:11.3f} grader")
print(f"Breddegrad (Latitude):   {kaffeklubben_geod_GRS[1]:11.3f} grader")
print(f"Højde:                   {kaffeklubben_geod_GRS[2]:11.3f} [m]")

9.5.1
/usr/local/lib/python3.12/dist-packages/pyproj/proj_dir/share/proj
Nye koordinater:
Længdegrad (longitude):      -30.510 grader
Breddegrad (Latitude):        83.671 grader
Højde:                        27.835 [m]


### 1.b Transformer kaffeklubben til geodætiske koordinater med middelhavniveau som højdereference
Vi transformerer koordinaterne fra kaffeklubben

In [13]:
ellipsoide = 'GRS80' # OBS: skal være skrevet med stort!
fra_koordinat_system = 'utm' # OBS: skal være skrevet med småt!
zone = '24'
grid = 'gvr2016'

"""
UDFYLD IKKE NEDENSTÅENDE
"""


pipeline_v2 = f"+ellps={ellipsoide} +proj=pipeline +step +proj={fra_koordinat_system} +zone={zone} +inv +step +proj=vgridshift +grids={pyproj_dir}/dk_sdfe_{grid}.tif"
transform_object = Transformer.from_pipeline(pipeline_v2)
kaffeklubben_new = transform_object.transform(*kaffeklubben)

print("Nye koordinater:")
print(f"Længdegrad (longitude):  {kaffeklubben_new[0]:11.3f} grader")
print(f"Breddegrad (Latitude):   {kaffeklubben_new[1]:11.3f} grader")
print(f"Højde:                   {kaffeklubben_new[2]:11.3f} [m]")

Nye koordinater:
Længdegrad (longitude):      -30.510 grader
Breddegrad (Latitude):        83.671 grader
Højde:                         0.594 [m]


### 1.c Transformer kaffeklubben til kartesiske koordinater

In [14]:
# udfyld nedenstående ukendte for at lave transformationen
ellipsoide = 'GRS80' # OBS: skal være skrevet med stort!
fra_koordinat_system = 'utm' # OBS: skal være skrevet med småt!
zone = '24'
til_koordinat_system = 'cart' # OBS: skal være skrevet med småt!

"""
UDFYLD IKKE NEDENSTÅENDE
"""

# input til proj for at lave transformationen
pipeline_v3 = f"+ellps={ellipsoide} +proj=pipeline +step +proj={fra_koordinat_system} +zone={zone} +inv +step +proj={til_koordinat_system}"
transform_object = Transformer.from_pipeline(pipeline_v3)
kaffeklubben_cart = transform_object.transform(*kaffeklubben)

# print for at se resultaterne
print("Nye koordinater:")
print(f"X:   {kaffeklubben_cart[0]:11.3f} [m]")
print(f"Y:   {kaffeklubben_cart[1]:11.3f} [m]")
print(f"Z:   {kaffeklubben_cart[2]:11.3f} [m]")

Nye koordinater:
X:    607788.588 [m]
Y:   -358151.593 [m]
Z:   6317776.980 [m]


Som I kan se er "Z" næsten lige med jordens radius (6371 km), hvordan kan det være? Hvad forventer vi at "Z" er i Danmark og ved ækvator?

### OPGAVE 2 - Kartesiske koordinater i Sisimiut
I har været ude og måle flg kartesiske koordinater i Sisimiut i GR96 over et fiks-punkt

| X    | Y    | Z |
| ---        | ---         | ---             |
| 1485712.083 | -2017839.628 | 5845771.729 |

### 2.a Transformer koordinaterne til UTM22N (GVR2016)

In [15]:
fikspunkt = [1485712.083, -2017839.628, 5845771.729]  # <----- indsæt her

# udfyld nedenstående ukendte for at lave transformationen
ellipsoide = 'GRS80' # OBS: skal være skrevet med stort!
fra_koordinat_system = 'cart' # OBS: skal være skrevet med småt!
grid_skifte_reference = 'gvr2016' # OBS: skal være skrevet med småt!
til_koordinat_system = 'utm' # OBS: skal være skrevet med småt!
zone = '22'

"""
UDFYLD IKKE NEDENSTÅENDE
"""

# input til proj for at lave transformationen
pipeline_v4 = f"+ellps={ellipsoide} +proj=pipeline +step +proj={fra_koordinat_system} +inv +step +proj=vgridshift +grids=dk_sdfe_{grid_skifte_reference}.tif +step +proj={til_koordinat_system} +zone={zone}"
transform_object2 = Transformer.from_pipeline(pipeline_v4)
fikspunkt_UTM = transform_object2.transform(*fikspunkt)

# print for at se resultaterne
print("Koordinater i UTM 22N:")
print(f"Easting:    {fikspunkt_UTM[0]:11.3f} [m]")
print(f"Northing:   {fikspunkt_UTM[1]:11.3f} [m]")
print(f"Height:     {fikspunkt_UTM[2]:11.3f} [m]")

Koordinater i UTM 22N:
Easting:     384783.599 [m]
Northing:   7426750.135 [m]
Height:         109.441 [m]


### 2.b Hvilket fikspunkt er det?
Tjek [Valdemar](https://valdemar.kortforsyningen.dk)

### OPGAVE 3 - DTU Campus Ballerup

Om to år skal I til Ballerup, her er de geodætiske koordinater til Ballerup campus i grader ([GRADER]d[MINUT]'[SEKUND]")

| Længdegrad    | Breddegrad    | Ellepsioidehøjde |
| ---        | ---         | ---             |
| 12d23'52"E | 55d43'52"N | 61         |


### 3.a Transformer koordinater fra grader til decimalgrader
Nedenfor er der et eksempel I kan rette i

In [16]:
# Koordinaterne er
# længdegrad: 11d12'13"E
# breddegrad: 14d15'16"N
# højde = 20
længdegrad_grader = 12     # <------ Ret her
længdegrad_minut = 23      # <------ Ret her
længdegrad_sekund = 52     # <------ Ret her

breddegrad_grader = 55     # <------ Ret her
breddegrad_minut = 43      # <------ Ret her
breddegrad_sekund = 52     # <------ Ret her

højde = 61                 # <------ Ret her


længdegrad_decimal_grader = længdegrad_grader + længdegrad_minut/60 + længdegrad_sekund/(60*60)
breddegrad_decimal_grader = breddegrad_grader + breddegrad_minut/60 + breddegrad_sekund/(60*60)

DTU_ballerup = [længdegrad_decimal_grader, breddegrad_decimal_grader, højde]

print("Koordinater i decimalgrader:")
print(f"Længdegrad (longitude):  {DTU_ballerup[0]:11.3f} grader")
print(f"Breddegrad (Latitude):   {DTU_ballerup[1]:11.3f} grader")
print(f"Højde:                   {DTU_ballerup[2]:11.3f} [m]")

Koordinater i decimalgrader:
Længdegrad (longitude):       12.398 grader
Breddegrad (Latitude):        55.731 grader
Højde:                        61.000 [m]


### 3.b Transformer koordinaterne til UTM 32N (dvr90)

In [17]:
# udfyld nedenstående ukendte for at lave transformationen
ellipsoide = 'GRS80' # OBS: skal være skrevet med stort!
grid_skifte_reference = "dvr90" #'dvr90' # OBS: skal være skrevet med småt!
til_koordinat_system = 'utm' # OBS: skal være skrevet med småt!
zone = '32'

"""
UDFYLD IKKE NEDENSTÅENDE
"""

# input til proj for at lave transformationen
pipeline_v5 = f"+ellps={ellipsoide} +proj=pipeline +step +proj=vgridshift +grids=dk_sdfe_{grid_skifte_reference}.tif +step +proj={til_koordinat_system} +zone={zone}"
transform_object = Transformer.from_pipeline(pipeline_v5)
DTU_ballerup_UTM32N = transform_object.transform(*DTU_ballerup)


print("Koordinater i UTM 32N:") # <--- Indsæt her
print(f"Easting:    {DTU_ballerup_UTM32N[0]:11.3f} [m]") # <--- Indsæt her
print(f"Northing:   {DTU_ballerup_UTM32N[1]:11.3f} [m]") # <--- Indsæt her
print(f"Height:     {DTU_ballerup_UTM32N[2]:11.3f} [m]") # <--- Indsæt her

Koordinater i UTM 32N:
Easting:     713335.175 [m]
Northing:   6181383.691 [m]
Height:          24.838 [m]
